## Simple RAG
Here we can use the Python SDK to develop a simple RAG agent with Milvus retrievers, then save the agent to a config.yaml and run it from there.

**Prerequisites:**
- Milvus server running at `localhost:19530`
- Collections named `cuda_docs` and `mcp_docs` with embedded documents

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath("../../../src/")
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
from pydantic import HttpUrl

from nat.agent.sdk import NatReActAgent
from nat.embedder.sdk import NIMEmbedder
from nat.llm.sdk import NimLLM
from nat.retriever.sdk import MilvusRetriever
from nat.tool.sdk import NatRetrieverTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0,
    max_tokens=4096,
    top_p=1.0,
    name="nim_llm",
)

milvus_embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    truncate="END",
    name="milvus_embedder",
)

cuda_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="cuda_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="cuda_retriever",
)

mcp_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="mcp_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="mcp_retriever",
)

cuda_retriever_tool = NatRetrieverTool(
    nat_retriever=cuda_retriever,
    topic="Retrieve documentation for NVIDIA's CUDA library",
    name="cuda_retriever_tool",
)

mcp_retriever_tool = NatRetrieverTool(
    nat_retriever=mcp_retriever,
    topic="Retrieve information about Model Context Protocol (MCP)",
    name="mcp_retriever_tool",
)

agent = NatReActAgent(
    tools=[cuda_retriever_tool, mcp_retriever_tool],
    llm=llm,
    verbose=True,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)


In [ ]:
await nat_workflow.prompt('How do I install CUDA?')

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'nat.utils.sdk.nat_evaluator'

In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

'To install CUDA, follow the steps outlined above, and make sure to consult the official NVIDIA CUDA documentation for detailed instructions tailored to your specific system configuration.'

functions:
  cuda_retriever_tool:
    retriever: cuda_retriever
    topic: Retrieve documentation for NVIDIA's CUDA library
    _type: nat_retriever
  mcp_retriever_tool:
    retriever: mcp_retriever
    topic: Retrieve information about Model Context Protocol (MCP)
    _type: nat_retriever
  add_memory_tool:
    description: |-
      Add any facts about user preferences to long term memory. Always use this if users mention a preference.
      The input to this tool should be a string that describes the user's preference, not the question or answer.
    memory: saas_memory
    _type: add_memory
  get_memory_tool:
    description: |-
      Always call this tool before calling any other tools, even if the user does not mention to use it.
      The question should be about user preferences which will help you format your response.
      For example: "How does the user like responses formatted?
    memory: saas_memory
    _type: get_memory
llms:
  nim_llm:
    model: nvdev/meta/llama-3.3-7